# 03. ETL INEGI-ATUS — Avance 03

**Proyecto:** Análisis de hechos de tránsito en la Ciudad de México  
**Equipo:** A&C — Méndez Cabral Ángel David (`23-003-1466`) y Ruiz Moreno Citlalli Viviana (`23-003-1162`) · UACM · Inteligencia de Negocios · Grupo 301 · 2026-2

## Objetivo
Procesar los registros de **INEGI-ATUS (fuente de referencia estadística, 2018–2025)** sin modificar los datos crudos (`RG-01`). Evaluamos duplicados exactos estrictamente sobre las **46 columnas oficiales** de ATUS (`INEGI-R01`), validamos la fecha calendárica reconstruida a partir de `ANIO`, `MES` e `ID_DIA` (`INEGI-R02`), verificamos consistencia territorial contra las 16 alcaldías de CDMX (`INEGI-R03`) y validamos los conteos enteros de personas heridas y fallecidas (`RG-03` / `RG-10`).

**Pregunta de negocio vigente:** ¿En qué alcaldías y franjas horarias de la Ciudad de México se registra la mayor cantidad de personas lesionadas en hechos de tránsito, de acuerdo con los registros disponibles a partir de 2018?

## 1. Entorno, rutas, modo de lectura y bitácora automática
Localiza automáticamente `resultados/resultados_INEGI_ATUS/etapa3/INEGI_ATUS_etapa3_conservados.csv` (Modo A) o los archivos anuales `atus_anual_2018..2025.csv` + `tc_municipio.csv` en `CARPETA_DATOS/INEGI/` en solo lectura (Modo B). Configura exportación verificada en `.csv.gz` y sincronización automática con `bitacora/bitacora.csv`.

In [1]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import csv
import gzip
import hashlib
import re
import unicodedata
import numpy as np
import pandas as pd
from IPython.display import display

TZ = ZoneInfo("America/Mexico_City")
RESPONSABLE_EQUIPO = "Equipo A&C (Méndez Cabral Ángel David; Ruiz Moreno Citlalli Viviana)"
VERSION_PROCESO = "Avance03-v4"

def localizar_raiz_avance03():
    actual = Path.cwd().resolve()
    for carpeta in [actual, *actual.parents]:
        if (carpeta / "03_ETL_INEGI_ATUS_Avance03 .ipynb").is_file() or carpeta.name.lower() == "avance03":
            return carpeta
    return actual

ROOT = localizar_raiz_avance03()
RUN_ID = datetime.now(TZ).strftime("%Y%m%d_%H%M%S")
OUT = ROOT / "data" / "processed" / "INEGI_ATUS" / RUN_ID
LOG_DIR = ROOT / "bitacora" / "INEGI_ATUS" / RUN_ID
BITACORA_GLOBAL_PATH = ROOT / "bitacora" / "bitacora.csv"
OUT.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

MUERTOS_ATUS = ["CONDMUERTO", "PASAMUERTO", "PEATMUERTO", "CICLMUERTO", "OTROMUERTO", "NEMUERTO"]
HERIDOS_ATUS = ["CONDHERIDO", "PASAHERIDO", "PEATHERIDO", "CICLHERIDO", "OTROHERIDO", "NEHERIDO"]
COLS_AUX_AVANCE02 = {
    "archivo_origen", "fila_origen", "alcaldia_normalizada", "motivo_exclusion",
    "huella_fila_original", "grupo_identico_original", "duplicado_adicional_original",
    "fecha_validada", "fecha_calendario_invalida",
    "total_fallecidos_validado", "total_lesionados_validado",
    *(f"{c}_valido" for c in MUERTOS_ATUS + HERIDOS_ATUS),
}

ATUS_ETAPA3 = None
CARPETA_CRUDA_INEGI = None
for carpeta in [ROOT, *ROOT.parents, ROOT.parent.parent]:
    cand_et3 = carpeta / "resultados" / "resultados_INEGI_ATUS" / "etapa3" / "INEGI_ATUS_etapa3_conservados.csv"
    if cand_et3.is_file():
        ATUS_ETAPA3 = cand_et3
        break
    cand_raw = carpeta / "CARPETA_DATOS" / "INEGI" / "conjunto_de_datos_atus_anual_csv"
    if CARPETA_CRUDA_INEGI is None and (cand_raw / "catalogos" / "tc_municipio.csv").is_file():
        CARPETA_CRUDA_INEGI = cand_raw
    cand_raw2 = carpeta / "INEGI" / "conjunto_de_datos_atus_anual_csv"
    if CARPETA_CRUDA_INEGI is None and (cand_raw2 / "catalogos" / "tc_municipio.csv").is_file():
        CARPETA_CRUDA_INEGI = cand_raw2

if ATUS_ETAPA3 is not None:
    MODO_ENTRADA = "etapa3_avance02"
    ORIGEN_INFO = ATUS_ETAPA3
elif CARPETA_CRUDA_INEGI is not None:
    MODO_ENTRADA = "crudos_carpeta_datos"
    ORIGEN_INFO = CARPETA_CRUDA_INEGI
else:
    raise FileNotFoundError("No se localizaron ni 'INEGI_ATUS_etapa3_conservados.csv' ni 'CARPETA_DATOS/INEGI/conjunto_de_datos_atus_anual_csv/'.")

BITACORA = []

def loggear(archivo, paso, regla, antes, despues, resultado, detalle="", observaciones="", responsable=RESPONSABLE_EQUIPO):
    BITACORA.append({
        "fecha": datetime.now(TZ).isoformat(timespec="seconds"),
        "fuente": "INEGI_ATUS",
        "run_id": RUN_ID,
        "responsable": responsable,
        "archivo": str(archivo),
        "paso": paso,
        "regla": regla,
        "filas_antes": int(antes),
        "filas_despues": int(despues),
        "resultado": resultado,
        "detalle": detalle,
        "observaciones": observaciones,
        "version_proceso": VERSION_PROCESO,
    })

def sincronizar_bitacora():
    df_run = pd.DataFrame(BITACORA)
    BITACORA_GLOBAL_PATH.parent.mkdir(parents=True, exist_ok=True)
    if BITACORA_GLOBAL_PATH.is_file():
        df_prev = pd.read_csv(BITACORA_GLOBAL_PATH, dtype=str, encoding="utf-8-sig")
        if {"fuente", "run_id"}.issubset(df_prev.columns):
            df_prev = df_prev.loc[~(df_prev["fuente"].eq("INEGI_ATUS") & df_prev["run_id"].eq(RUN_ID))]
        df_all = pd.concat([df_prev, df_run.astype(str)], ignore_index=True)
    else:
        df_all = df_run
    df_all.to_csv(BITACORA_GLOBAL_PATH, index=False, encoding="utf-8-sig")
    df_run.to_csv(LOG_DIR / "bitacora_INEGI_ATUS.csv", index=False, encoding="utf-8-sig")
    return df_run

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1024 * 1024), b""):
            h.update(b)
    return h.hexdigest()

def escribir(df, path):
    path = Path(path)
    if not path.name.endswith(".csv.gz"):
        if path.suffix == ".csv":
            path = Path(str(path) + ".gz")
        else:
            path = Path(str(path) + ".csv.gz")
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        with gzip.open(path, "wt", encoding="utf-8-sig", newline="") as gz:
            df.to_csv(gz, index=False, chunksize=10000, quoting=csv.QUOTE_MINIMAL)
        with gzip.open(path, "rt", encoding="utf-8-sig", newline="") as gz:
            lector = csv.reader(gz)
            next(lector, None)
            filas = sum(1 for _ in lector)
        assert filas == len(df), f"Conteo incorrecto en {path.name}: {filas} != {len(df)}"
        return path
    except Exception:
        if path.exists():
            path.unlink()
        raise

print("Raíz Avance 03:", ROOT)
print("Modo entrada ATUS:", MODO_ENTRADA)
print("Origen ATUS:", ORIGEN_INFO)
print("RUN_ID:", RUN_ID)
print("Salida:", OUT)

Raíz Avance 03: C:\Users\angel\Downloads\CARPETA_DATOS
Modo entrada ATUS: crudos_carpeta_datos
Origen ATUS: C:\Users\angel\Downloads\CARPETA_DATOS\CARPETA_DATOS\INEGI\conjunto_de_datos_atus_anual_csv
RUN_ID: 20261109_043449
Salida: C:\Users\angel\Downloads\CARPETA_DATOS\data\processed\INEGI_ATUS\20261109_043449


## 2. Extracción, delimitación reproducible CDMX (2018–2025) e inventario
Leemos las claves oficiales de alcaldía desde `tc_municipio.csv` (`CVE_ENT = '09'`, `CVE_MUN != '999'`) cuando ejecutamos desde crudos, delimitando por bloques sin saturar la memoria y conservando intactas las 46 columnas originales de ATUS.

In [2]:
inventario = []

if MODO_ENTRADA == "etapa3_avance02":
    df_atus = pd.read_csv(ORIGEN_INFO, dtype="string", encoding="utf-8-sig", low_memory=False)
    cols_orig_46 = [c for c in df_atus.columns if c not in COLS_AUX_AVANCE02]
    assert len(cols_orig_46) == 46, f"Se esperaban 46 cols originales; encontradas {len(cols_orig_46)}"
    h = sha256(ORIGEN_INFO)
    inventario.append({
        "archivo": ORIGEN_INFO.name,
        "ruta": str(ORIGEN_INFO),
        "filas_crudo": len(df_atus),
        "filas_delimitadas": len(df_atus),
        "columnas_originales": len(cols_orig_46),
        "columnas_totales": len(df_atus.columns),
        "sha256": h,
    })
    loggear(
        ORIGEN_INFO.name, "Extracción etapa3", "RG-01/RG-11",
        len(df_atus), len(df_atus), "Correcto; 46 cols originales identificadas",
        f"Cols entrada={len(df_atus.columns)}; Cols originales={len(cols_orig_46)}; SHA256={h}",
        "Se separan las 23 columnas auxiliares previas para evaluar duplicidad estricta."
    )
    nombre_tabla = ORIGEN_INFO.name
else:
    cat_path = ORIGEN_INFO / "catalogos" / "tc_municipio.csv"
    cat = pd.read_csv(cat_path, dtype="string", encoding="utf-8-sig")
    cdmx_cat = cat.loc[(cat["CVE_ENT"] == "09") & (cat["CVE_MUN"] != "999")].copy()
    claves_cdmx = set(cdmx_cat["CVEGEO"].str.zfill(5))
    mapa_alcaldias = dict(zip(cdmx_cat["CVEGEO"].str.zfill(5), cdmx_cat["NOM_MUNICIPIO"]))
    assert len(claves_cdmx) == 16

    partes_cdmx = []
    cols_orig_46 = None
    for anio in range(2018, 2026):
        fpath = ORIGEN_INFO / "conjunto_de_datos" / f"atus_anual_{anio}.csv"
        h = sha256(fpath)
        n_raw = 0
        bloques_ok = []
        for bloque in pd.read_csv(fpath, dtype="string", encoding="utf-8-sig", chunksize=50000, low_memory=False):
            n_raw += len(bloque)
            if cols_orig_46 is None:
                cols_orig_46 = list(bloque.columns)
            ent = bloque["CVE_ENT"].fillna("").str.strip().str.zfill(2)
            geo = bloque["CVEGEO"].fillna("").str.strip().str.zfill(5)
            mun = bloque["CVE_MUN"].fillna("").str.strip().str.zfill(3)
            yr = pd.to_numeric(bloque["ANIO"], errors="coerce")
            mask_ok = yr.between(2018, 2025) & ent.eq("09") & geo.isin(claves_cdmx) & mun.eq(geo.str[-3:])
            if mask_ok.any():
                sub = bloque.loc[mask_ok].copy()
                sub["archivo_origen"] = fpath.name
                sub["alcaldia_normalizada"] = geo.loc[mask_ok].map(mapa_alcaldias)
                bloques_ok.append(sub)
        df_y = pd.concat(bloques_ok, ignore_index=True) if bloques_ok else pd.DataFrame(columns=cols_orig_46)
        partes_cdmx.append(df_y)
        inventario.append({
            "archivo": fpath.name,
            "ruta": str(fpath),
            "filas_crudo": n_raw,
            "filas_delimitadas": len(df_y),
            "columnas_originales": len(cols_orig_46),
            "columnas_totales": len(df_y.columns),
            "sha256": h,
        })
        loggear(
            fpath.name, f"Extracción y delimitación CDMX ({anio})", "RG-01/INEGI-R03",
            n_raw, len(df_y), "Correcto; delimitado con catálogo oficial INEGI",
            f"Excluidas nacional/fuera CDMX={n_raw - len(df_y):,}; SHA256={h}",
            "Archivos anuales crudos intactos en CARPETA_DATOS/INEGI/."
        )
    df_atus = pd.concat(partes_cdmx, ignore_index=True)
    assert len(cols_orig_46) == 46
    nombre_tabla = "INEGI_ATUS"

inventario_df = pd.DataFrame(inventario)
display(inventario_df)
escribir(inventario_df, OUT / "inventario_fuentes.csv.gz")
tablas = {nombre_tabla: df_atus}

,archivo,ruta,filas_crudo,filas_delimitadas,columnas_originales,columnas_totales,sha256
0,atus_anual_2018.csv,C:\Users\angel\Downloads\CARPETA_DATOS\CARPETA_DATOS\INEGI\conjunto_de_datos_atus_anual_csv\conjunto_de_datos\atus_anual_2018.csv,365281,11656,46,48,071a1b28a7c831b29420a20b37da1406c72ad06ab04c3daf3b41bec4729a6ac8
1,atus_anual_2019.csv,C:\Users\angel\Downloads\CARPETA_DATOS\CARPETA_DATOS\INEGI\conjunto_de_datos_atus_anual_csv\conjunto_de_datos\atus_anual_2019.csv,362729,10673,46,48,514d4c7f30958718c4ef5b79036d98ee39b89159d52364f47040fb3611be88cb
2,atus_anual_2020.csv,C:\Users\angel\Downloads\CARPETA_DATOS\CARPETA_DATOS\INEGI\conjunto_de_datos_atus_anual_csv\conjunto_de_datos\atus_anual_2020.csv,301678,6549,46,48,4cd4565831108003591ca591e71d2c460603c64de368e25285f27436dda0f225
3,atus_anual_2021.csv,C:\Users\angel\Downloads\CARPETA_DATOS\CARPETA_DATOS\INEGI\conjunto_de_datos_atus_anual_csv\conjunto_de_datos\atus_anual_2021.csv,340415,6743,46,48,fbc197c16b975f058156122f2a1b5ce57685a655dc47171953b2ade6801a0712
4,atus_anual_2022.csv,C:\Users\angel\Downloads\CARPETA_DATOS\CARPETA_DATOS\INEGI\conjunto_de_datos_atus_anual_csv\conjunto_de_datos\atus_anual_2022.csv,377231,7596,46,48,bffa7d5e817e97d3cd0150a806463f7ae3647fdb4327f76436361f7b16e0d7ba
5,atus_anual_2023.csv,C:\Users\angel\Downloads\CARPETA_DATOS\CARPETA_DATOS\INEGI\conjunto_de_datos_atus_anual_csv\conjunto_de_datos\atus_anual_2023.csv,381048,7276,46,48,121d7743238ce725eb6898fe548618d4d60e347dc1438b941060b4201b33cc22
6,atus_anual_2024.csv,C:\Users\angel\Downloads\CARPETA_DATOS\CARPETA_DATOS\INEGI\conjunto_de_datos_atus_anual_csv\conjunto_de_datos\atus_anual_2024.csv,374949,6655,46,48,00bd5292cff42b13edd1e9c828e898890073e8d729eeee405de418d70263d29c
7,atus_anual_2025.csv,C:\Users\angel\Downloads\CARPETA_DATOS\CARPETA_DATOS\INEGI\conjunto_de_datos_atus_anual_csv\conjunto_de_datos\atus_anual_2025.csv,380391,5276,46,48,6bc21394ea023cb75752d3741f8c828ec99cbaacde410d5b9cc0638140524fed


## 3. Duplicados exactos sobre las 46 columnas originales (`INEGI-R01`)
Se comparan exclusivamente las **46 columnas originales** de ATUS (`keep="first"`). Esto permite detectar las **145 copias exactas adicionales** (que quedaban ocultas si se comparaban columnas derivadas como `fila_origen`), separándolas del conjunto limpio sin perder evidencia. Se documenta además la ausencia de un folio operativo único en la fuente ATUS.

In [3]:
resultados = {}
RESUMEN_DUPLICADOS = []

for nombre, original in tablas.items():
    entrada = len(original)

    # INEGI-R01: duplicados exactos estrictamente en las 46 columnas originales
    mask_dup = original.duplicated(subset=cols_orig_46, keep="first")
    duplicados = original.loc[mask_dup].copy()
    base = original.loc[~mask_dup].copy()

    if len(duplicados):
        duplicados["_motivo_separacion"] = "Copia exacta adicional en 46 columnas originales de ATUS"
        duplicados["_regla_aplicada"] = "INEGI-R01/RG-04"

    # Revisar si existen fechas inválidas dentro de los duplicados exactos retirados (auditoría)
    if len(duplicados):
        ftxt_dup = (
            duplicados["ANIO"].fillna("").str.strip() + "-" +
            duplicados["MES"].fillna("").str.strip().str.zfill(2) + "-" +
            duplicados["ID_DIA"].fillna("").str.strip().str.zfill(2)
        )
        n_fech_inv_en_dups = int(pd.to_datetime(ftxt_dup, format="%Y-%m-%d", errors="coerce").isna().sum())
    else:
        n_fech_inv_en_dups = 0

    folios_repetidos = base.iloc[0:0].copy()

    resultados[nombre] = {
        "base": base,
        "duplicados": duplicados,
        "folios_repetidos": folios_repetidos,
        "entrada": entrada,
        "n_fech_inv_en_dups": n_fech_inv_en_dups,
    }

    RESUMEN_DUPLICADOS.append({
        "archivo": nombre,
        "registros_delimitados_entrada": entrada,
        "columnas_originales_evaluadas": len(cols_orig_46),
        "duplicados_exactos_retirados": len(duplicados),
        "duplicados_con_fecha_invalida": n_fech_inv_en_dups,
        "registros_unicos": len(base),
        "columna_folio": "No aplica (fuente ATUS sin folio único)",
    })

    loggear(
        nombre, "Duplicados exactos (46 cols)", "INEGI-R01/RG-04",
        entrada, len(base), f"Retirados {len(duplicados)} duplicados exactos",
        f"Cols evaluadas=46; de las 3 filas 2023-12-32 previas, {n_fech_inv_en_dups} era copia exacta duplicada",
        "La ausencia de folio único en ATUS se documenta explícitamente; folios repetidos = 0."
    )

resumen_duplicados_df = pd.DataFrame(RESUMEN_DUPLICADOS)
display(resumen_duplicados_df)

,archivo,registros_delimitados_entrada,columnas_originales_evaluadas,duplicados_exactos_retirados,duplicados_con_fecha_invalida,registros_unicos,columna_folio
0,INEGI_ATUS,62424,46,145,1,62279,No aplica (fuente ATUS sin folio único)


## 4. Validación calendárica (`INEGI-R02`), territorial (`INEGI-R03`) y víctimas
Construimos la fecha calendárica a partir de **`ANIO`, `MES` e `ID_DIA`** en formato estricto `%Y-%m-%d` sin alterar las columnas originales. Los días imposibles (por ejemplo `ID_DIA = 32` en `2023-12-32`) pasan a **`pendiente`** (`INEGI-R02`) en vez de corregirse por suposición. Adicionalmente validamos `ID_HORA` (`0–23`), asignamos franja horaria auxiliar y validamos enteros `>= 0` en los 12 campos de víctimas (`CONDMUERTO..NEHERIDO`).

In [4]:
RESUMEN_VALIDACION = []

for nombre, r in resultados.items():
    base = r["base"].copy()
    base["_decision"] = "conservado"
    base["_motivo_decision"] = "Cumple validaciones calendáricas, territoriales y de víctimas"
    base["_regla_aplicada"] = "RG-10"
    base["_valores_ausentes"] = base[cols_orig_46].isna().sum(axis=1).astype("int64")

    # INEGI-R02: Fecha calendárica desde ANIO, MES, ID_DIA
    anio_s = base["ANIO"].fillna("").str.strip()
    mes_s = base["MES"].fillna("").str.strip().str.zfill(2)
    dia_s = base["ID_DIA"].fillna("").str.strip().str.zfill(2)
    fecha_txt = anio_s + "-" + mes_s + "-" + dia_s
    fecha_conv = pd.to_datetime(fecha_txt, format="%Y-%m-%d", errors="coerce")

    base["_fecha_validada"] = fecha_conv.dt.strftime("%Y-%m-%d").fillna("")
    base["_fecha_calendario_invalida"] = fecha_conv.isna()
    invalidas_unicas = int(fecha_conv.isna().sum())

    base.loc[fecha_conv.isna(), "_decision"] = "pendiente"
    base.loc[fecha_conv.isna(), "_motivo_decision"] = "Fecha calendárica imposible (ANIO/MES/ID_DIA, ej. día 32); no se imputa"
    base.loc[fecha_conv.isna(), "_regla_aplicada"] = "INEGI-R02"

    # Horas y franja horaria auxiliar (RG-10 / INEGI-R04)
    hora_num = pd.to_numeric(base["ID_HORA"].fillna("").str.strip(), errors="coerce")
    hora_ok = hora_num.between(0, 23).fillna(False)
    base["_hora_valida"] = hora_num.where(hora_ok)
    base["_franja_horaria"] = pd.cut(
        base["_hora_valida"], bins=[-1, 5, 11, 17, 23],
        labels=["Madrugada", "Mañana", "Tarde", "Noche"]
    ).astype("string").fillna("")

    # Víctimas heridas y fallecidas (RG-03 / RG-10)
    h_df = base[HERIDOS_ATUS].apply(pd.to_numeric, errors="coerce")
    m_df = base[MUERTOS_ATUS].apply(pd.to_numeric, errors="coerce")
    h_ok = h_df.notna().all(axis=1) & (h_df >= 0).all(axis=1) & (h_df % 1 == 0).all(axis=1)
    m_ok = m_df.notna().all(axis=1) & (m_df >= 0).all(axis=1) & (m_df % 1 == 0).all(axis=1)

    base["_total_heridos_validado"] = h_df.sum(axis=1).where(h_ok).astype("Int64")
    base["_total_fallecidos_validado"] = m_df.sum(axis=1).where(m_ok).astype("Int64")

    tot_heridos_unicos = int(base["_total_heridos_validado"].dropna().sum())
    tot_heridos_conservados = int(base.loc[base["_decision"].eq("conservado"), "_total_heridos_validado"].dropna().sum())
    tot_fallecidos_unicos = int(base["_total_fallecidos_validado"].dropna().sum())

    RESUMEN_VALIDACION.append({
        "archivo": nombre,
        "registros_unicos_evaluados": len(base),
        "fechas_invalidas_en_unicos_pendientes": invalidas_unicas,
        "fechas_invalidas_totales_pre_dedup": invalidas_unicas + r["n_fech_inv_en_dups"],
        "horas_fuera_0_23": int((~hora_ok).sum()),
        "conservados": int(base["_decision"].eq("conservado").sum()),
        "pendientes": int(base["_decision"].eq("pendiente").sum()),
        "excluidos": int(base["_decision"].eq("excluido").sum()),
        "total_heridos_unicos": tot_heridos_unicos,
        "total_heridos_conservados": tot_heridos_conservados,
        "total_fallecidos_unicos": tot_fallecidos_unicos,
    })

    loggear(
        nombre, "Validación calendario y víctimas (INEGI-R02)", "INEGI-R02/INEGI-R03",
        len(base), int(base["_decision"].eq("conservado").sum()),
        f"Conservados={int(base['_decision'].eq('conservado').sum()):,}; Pendientes={invalidas_unicas}",
        (
            f"Fechas imposibles (2023-12-32): {invalidas_unicas} únicas + {r['n_fech_inv_en_dups']} en dups = "
            f"{invalidas_unicas + r['n_fech_inv_en_dups']} totales; Heridos únicos={tot_heridos_unicos:,}"
        ),
        "No se corrige el día 32 por suposición; las filas permanecen separadas en pendientes."
    )
    r["base"] = base

resumen_validacion_df = pd.DataFrame(RESUMEN_VALIDACION)
display(resumen_validacion_df)
print("\nDetalle de filas únicas con fecha calendárica inválida (pendientes):")
display(base.loc[base["_fecha_calendario_invalida"], ["ANIO", "MES", "ID_DIA", "CVEGEO", "alcaldia_normalizada", "_total_heridos_validado", "_total_fallecidos_validado"]])


Detalle de filas únicas con fecha calendárica inválida (pendientes):


,archivo,registros_unicos_evaluados,fechas_invalidas_en_unicos_pendientes,fechas_invalidas_totales_pre_dedup,horas_fuera_0_23,conservados,pendientes,excluidos,total_heridos_unicos,total_heridos_conservados,total_fallecidos_unicos
0,INEGI_ATUS,62279,2,3,0,62277,2,0,17637,17637,1896


,ANIO,MES,ID_DIA,CVEGEO,alcaldia_normalizada,_total_heridos_validado,_total_fallecidos_validado
46202,2023,12,32,09007,Iztapalapa,0,0
49428,2023,12,32,09015,Cuauhtémoc,0,0


## 5. Exportación de resultados, conciliación y bitácora
Comprobamos la conciliación exacta sobre el universo delimitado CDMX 2018–2025:
$$	ext{62,424 delimitadas} = 	ext{62,277 conservados} + 	ext{2 pendientes (fecha 2023-12-32)} + 	ext{0 excluidos} + 	ext{145 duplicados exactos}$$

In [5]:
controles = []

for nombre, r in resultados.items():
    base = r["base"]
    conservados = base.loc[base["_decision"].eq("conservado")].copy()
    pendientes = base.loc[base["_decision"].eq("pendiente")].copy()
    excluidos = base.loc[base["_decision"].eq("excluido")].copy()
    duplicados = r["duplicados"]
    folios_rep = r["folios_repetidos"]

    prefijo = Path(nombre).stem
    escribir(conservados, OUT / f"{prefijo}_limpio.csv.gz")
    escribir(pendientes, OUT / f"{prefijo}_pendientes.csv.gz")
    escribir(excluidos, OUT / f"{prefijo}_excluidos_por_regla.csv.gz")
    escribir(duplicados, OUT / f"{prefijo}_duplicados_exactos.csv.gz")
    escribir(folios_rep, OUT / f"{prefijo}_folios_repetidos_revision.csv.gz")

    total = len(conservados) + len(pendientes) + len(excluidos) + len(duplicados)
    ok = total == r["entrada"]
    assert ok, f"Conciliación incorrecta en {nombre}: {total} != {r['entrada']}"

    controles.append({
        "archivo": nombre,
        "registros_iniciales_delimitados": r["entrada"],
        "conservados": len(conservados),
        "pendientes_fecha_invalida": len(pendientes),
        "excluidos": len(excluidos),
        "duplicados_exactos_46_cols": len(duplicados),
        "total_reconciliado": total,
        "conciliacion_correcta": ok,
    })

    loggear(
        nombre, "Exportación y conciliación", "RG-07/RG-11/RG-12/CV-15",
        r["entrada"], len(conservados), "Correcto; conciliación verificada (True)",
        f"Conservados={len(conservados):,}; Pendientes={len(pendientes)}; Excluidos=0; Dups={len(duplicados)}",
        "62,424 = 62,277 + 2 + 0 + 145."
    )

control_df = pd.DataFrame(controles)
display(control_df)
escribir(control_df, OUT / "conciliacion_por_archivo.csv.gz")
escribir(resumen_duplicados_df, OUT / "resumen_duplicados.csv.gz")
escribir(resumen_validacion_df, OUT / "resumen_validacion.csv.gz")

bitacora_df = sincronizar_bitacora()
display(bitacora_df[["paso", "archivo", "regla", "filas_antes", "filas_despues", "resultado"]].tail(10))

peso_total_mb = sum(p.stat().st_size for p in OUT.glob("*.csv.gz")) / (1024 ** 2)
print(f"\nETL INEGI-ATUS terminado en: {OUT} ({peso_total_mb:.2f} MB comprimidos)")
print(f"Bitácora consolidada actualizada en: {BITACORA_GLOBAL_PATH}")


ETL INEGI-ATUS terminado en: C:\Users\angel\Downloads\CARPETA_DATOS\data\processed\INEGI_ATUS\20261109_043449 (0.98 MB comprimidos)
Bitácora consolidada actualizada en: C:\Users\angel\Downloads\CARPETA_DATOS\bitacora\bitacora.csv


,archivo,registros_iniciales_delimitados,conservados,pendientes_fecha_invalida,excluidos,duplicados_exactos_46_cols,total_reconciliado,conciliacion_correcta
0,INEGI_ATUS,62424,62277,2,0,145,62424,True


,paso,archivo,regla,filas_antes,filas_despues,resultado
1,Extracción y delimitación CDMX (2019),atus_anual_2019.csv,RG-01/INEGI-R03,362729,10673,Correcto; delimitado con catálogo oficial INEGI
2,Extracción y delimitación CDMX (2020),atus_anual_2020.csv,RG-01/INEGI-R03,301678,6549,Correcto; delimitado con catálogo oficial INEGI
3,Extracción y delimitación CDMX (2021),atus_anual_2021.csv,RG-01/INEGI-R03,340415,6743,Correcto; delimitado con catálogo oficial INEGI
4,Extracción y delimitación CDMX (2022),atus_anual_2022.csv,RG-01/INEGI-R03,377231,7596,Correcto; delimitado con catálogo oficial INEGI
5,Extracción y delimitación CDMX (2023),atus_anual_2023.csv,RG-01/INEGI-R03,381048,7276,Correcto; delimitado con catálogo oficial INEGI
6,Extracción y delimitación CDMX (2024),atus_anual_2024.csv,RG-01/INEGI-R03,374949,6655,Correcto; delimitado con catálogo oficial INEGI
7,Extracción y delimitación CDMX (2025),atus_anual_2025.csv,RG-01/INEGI-R03,380391,5276,Correcto; delimitado con catálogo oficial INEGI
8,Duplicados exactos (46 cols),INEGI_ATUS,INEGI-R01/RG-04,62424,62279,Retirados 145 duplicados exactos
9,Validación calendario y víctimas (INEGI-R02),INEGI_ATUS,INEGI-R02/INEGI-R03,62279,62277,"Conservados=62,277; Pendientes=2"
10,Exportación y conciliación,INEGI_ATUS,RG-07/RG-11/RG-12/CV-15,62424,62277,Correcto; conciliación verificada (True)


## 6. Interpretación y limitaciones
- **Hallazgo clave de conciliación (`INEGI-R01` + `INEGI-R02`):** de las `3` filas originales con fecha imposible (`2023-12-32`) identificadas en Avance 02 sobre `62,424` filas, `1` fila era una copia exacta duplicada de otro registro (`INEGI-R01`), por lo que tras retirar los `145` duplicados exactos quedan **`2`** registros únicos en `pendientes` y **`62,277`** registros en el conjunto limpio (`conciliación = 62,424`).
- **Independencia metodológica:** INEGI-ATUS registra siniestros con menor volumen (`62,277` únicos en 2018–2025; `17,637` personas heridas únicas) que SSC o C5 debido a diferencias institucionales de captación; sirve únicamente como contraste estadístico y no se suma con otras fuentes.